In [22]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import m2cgen as m2c
import itertools
import random
import json
from pathlib import Path

RANDOM_STATE = 42
ROOT_DIR = Path.cwd().parent

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

def load_and_merge_files(file_paths, encoding="1251", sep=";"):
    """
    Читает файлы по списку путей и склеивает их в один DataFrame.
    """
    all_dfs = []
    
    for path in file_paths:
        print(f"Загрузка: {path}")
        # Читаем файл
        df = pd.read_csv(path, encoding=encoding, sep=sep)
        
        # ОЧЕНЬ ВАЖНО: очищаем названия колонок от пробелов сразу при загрузке
        df.columns = df.columns.str.strip()
        
        all_dfs.append(df)
    
    # Склеиваем всё вместе
    merged_df = pd.concat(all_dfs, axis=0, ignore_index=True, sort=False)
    
    print(f"Итоговый размер объединенного датасета: {merged_df.shape}")
    return merged_df

# Пример использования:
files = ["data/1.csv", "data/2.csv", "data/4.csv", "data/5.csv"]
df_train = load_and_merge_files(files)
df_train.columns = df_train.columns.str.strip()
print(df_train.columns)

# ==========================================
# 1. ОТНОСИТЕЛЬНЫЕ ПРИЗНАКИ (Исправленные)
# ==========================================
def extract_features(signal, window_size=15):
    """
    Вычисляет ОТНОСИТЕЛЬНЫЕ признаки.
    Смотрит на diff (разницу с предыдущей точкой) - это лучший маркер для скачков.
    """
    diff = signal.diff().abs().fillna(0)
    rolling_median = signal.rolling(window=window_size, min_periods=1).median().fillna(0)
    rolling_std = signal.rolling(window=window_size, min_periods=1).std().fillna(0)
    
    # Стабилизатор 0.1 - сглаживает микро-дребезг и защищает от деления на ноль
    base_level = np.abs(rolling_median) + 0.01 
    
    rel_diff = diff / base_level
    rel_std = rolling_std / base_level
    
    features = pd.DataFrame({
        'rel_diff': rel_diff.rolling(window=max(3, window_size//5), min_periods=1).mean(),
        'rel_std': rel_std
    })
    
    return features.bfill()

Загрузка: data/1.csv
Загрузка: data/2.csv
Загрузка: data/4.csv
Загрузка: data/5.csv
Итоговый размер объединенного датасета: (78211, 70)
Index(['TimeStamp', 'Температура масла в магистрали общей откачки (поз. Тм)',
       'Температура слива масла из опоры турбины (поз. Т606)',
       'Температура масла на входе в двигатель за фильтром (поз. Т607)',
       'Температура топливного газа перед СК (перед расходомерным устройством) (поз. Ттг)',
       'Давление масла в нагнетающей магистрали двигателя (после фильтра) (поз. Pм)',
       'Давление масла в магистрали общей откачки (поз. Р615)',
       'Давление за ОК Pk1', 'Давление за ОК Pk2',
       'Давление топливного газа перед дозаторами (поз. Pтг1.1)',
       'Вибрация промежуточного корпуса газогенератора (гориз.) (поз. В1)',
       'Вибрация корпуса силовой турбины (гориз.) (поз. В2)',
       'Положение дозатора газа 8402 (или ДУС-6,5 МП) ( поз. А ДГ)',
       'Контроль управления дозатором газа ДГ',
       'Частота вращения РНД (Расчет

In [23]:
df_val_test_raw = pd.read_csv("data/3.csv", encoding="1251", sep=";")
df_val_test_raw.columns = df_val_test_raw.columns.str.strip()

def create_corrupted_test_bench(df, time_column='ТР', corruption_ratio=0.3):
    """
    Создает копию данных с внедренными аномалиями.
    Исправлено: добавлена копия массива для обхода ошибки read-only.
    """
    df_corrupted: pd.DataFrame = df.copy()
    # Создаем матрицу меток (изначально всё 1.0)
    df_labels = pd.DataFrame(np.ones(df.shape), columns=df.columns, index=df.index)
    
    sensor_columns = [col for col in df.columns if col != time_column]
    
    # Случайный выбор колонок для порчи
    n_to_corrupt = int(len(sensor_columns) * corruption_ratio)
    target_cols = np.random.choice(sensor_columns, n_to_corrupt, replace=False)
    
    # print(f"Будут зашумлены следующие каналы: {list(target_cols)}")

    for col in sensor_columns:
        # ПРЕОБРАЗОВАНИЕ В ЧИСЛА
        df_corrupted[col] = pd.to_numeric(df_corrupted[col], errors='coerce').fillna(0).astype(float)
        
        if col in target_cols:
            # .copy() — ГЛАВНОЕ ИСПРАВЛЕНИЕ: создаем массив, доступный для записи
            signal = df_corrupted[col].values.copy()
            labels = df_labels[col].values.copy()
            
            base_level = np.abs(np.median(signal)) + 0.1
            n_pts = len(signal)
            
            # Внедряем одиночные скачки (Spikes)
            n_spikes = np.random.randint(10, 50)
            if n_pts > 10:
                spike_indices = np.random.choice(n_pts, n_spikes, replace=False)
                for idx in spike_indices:
                    magnitude = base_level * np.random.uniform(0.05, 0.15)
                    # Теперь запись разрешена
                    signal[idx] += np.random.choice([-1, 1]) * magnitude
                    labels[idx] = 0
            
            # Внедряем блоки дребезга (Noise Blocks)
            for _ in range(np.random.randint(2, 4)):
                block_len = np.random.randint(10, 30)
                if n_pts > block_len:
                    start_idx = np.random.randint(0, n_pts - block_len)
                    noise = np.random.normal(0, base_level * 0.05, block_len)
                    signal[start_idx : start_idx + block_len] += noise
                    labels[start_idx : start_idx + block_len] = 0
                
            # Возвращаем измененные данные обратно в DataFrame
            df_corrupted[col] = signal
            df_labels[col] = labels
        else:
            df_labels[col] = 1.0

    return df_corrupted, df_labels, target_cols


def split_datasets_by_sensors(df_values, df_labels, val_sensors, test_sensors):
    """
    Разбивает датафрейм с данными и датафрейм с метками на группы Val и Test.
    
    df_values: DataFrame с зашумленными данными
    df_labels: DataFrame с истинными метками (0/1)
    val_sensors: список названий колонок для валидации
    test_sensors: список названий колонок для теста
    """
    
    # 1. Проверяем, что все указанные датчики есть в данных (защита от ошибок)
    available_cols = df_values.columns.tolist()
    val_sensors = [c for c in val_sensors if c in available_cols]
    test_sensors = [c for c in test_sensors if c in available_cols]
    
    # 2. Создаем подмножества данных (с сохранением колонки времени для порядка)
    # Используем .copy(), чтобы избежать SettingWithCopyWarning
    df_val = df_values[val_sensors].copy()
    df_test = df_values[test_sensors].copy()
    
    # 3. Создаем подмножества меток
    labels_val = df_labels[val_sensors].copy()
    labels_test = df_labels[test_sensors].copy()
    
    print(f"✅ Разбиение завершено:")
    print(f"   - Валидация: {len(val_sensors)} датчиков")
    print(f"   - Тест:      {len(test_sensors)} датчиков")
    
    return df_val, labels_val, df_test, labels_test

# Генерируем данные
df_val_test, df_labels, corrupted_labels = create_corrupted_test_bench(df_val_test_raw, corruption_ratio=0.2)

# Определяем, какие потоки пойдут в val, а какие в test
stream_names = df_val_test.columns.to_list()[1:]
is_corrupted = [1 if col in corrupted_labels else 0 for col in stream_names]
val_streams, test_streams = train_test_split(stream_names, stratify=is_corrupted, test_size=0.5, random_state=RANDOM_STATE)

# Создаем размеченные val и test выборки
df_val, labels_val, df_test, labels_test = split_datasets_by_sensors(
    df_val_test, 
    df_labels, 
    val_streams, 
    test_streams
)


✅ Разбиение завершено:
   - Валидация: 20 датчиков
   - Тест:      20 датчиков


In [24]:
def compute_metrics(labels_true, labels_pred, dp=4) -> dict:
    metrics = dict()
    metrics["accuracy"] = round(accuracy_score(labels_true, labels_pred), dp)
    metrics["precision"] = round(precision_score(labels_true, labels_pred, pos_label=0, zero_division=0), dp)
    metrics["recall"] = round(recall_score(labels_true, labels_pred, pos_label=0, zero_division=0), dp)
    metrics["f1"] = round(f1_score(labels_true, labels_pred, pos_label=0, zero_division=0), dp)

    return metrics

In [25]:
def _train_model(df_train_raw, window_size, percentile_diff, max_depth,
                  corruption_ratio=0.3, time_column='TimeStamp', random_state=RANDOM_STATE):
    """
    Обучение модели на синтетических аномалиях.
    
    Ключевое отличие от предыдущей версии:
    - Признаки для НОРМАЛЬНЫХ точек считаются по чистому сигналу
    - Признаки для АНОМАЛЬНЫХ точек считаются по зашумлённому сигналу
    - Это устраняет проблему "размытия" аномалий rolling_median по зашумлённым данным
    """
    np.random.seed(random_state)
    
    sensor_cols = [c for c in df_train_raw.columns if c != time_column]
    
    # Генерируем аномалии — нужны зашумлённый сигнал + метки
    df_corrupted, df_labels, _ = create_corrupted_test_bench(
        df_train_raw,
        time_column=time_column,
        corruption_ratio=corruption_ratio
    )
    
    X_normal_list, X_anomaly_list = [], []
    
    for col in sensor_cols:
        # Чистый сигнал из оригинала
        clean_signal = pd.to_numeric(df_train_raw[col], errors='coerce').fillna(0).astype(float)
        # Зашумлённый сигнал
        noisy_signal = pd.to_numeric(df_corrupted[col], errors='coerce').fillna(0).astype(float)
        
        labels = pd.to_numeric(df_labels[col], errors='coerce').fillna(1).astype(int).values
        
        anomaly_mask  = (labels == 0)
        normal_mask   = (labels == 1)
        
        # Признаки нормальных точек — из ЧИСТОГО сигнала
        # rolling_median/std не загрязнены аномалиями → base_level стабилен
        feats_clean = extract_features(pd.Series(clean_signal), window_size).values
        X_normal_list.append(feats_clean[normal_mask])
        
        if anomaly_mask.any():
            # Признаки аномальных точек — из ЗАШУМЛЁННОГО сигнала
            # Именно так они будут выглядеть на инференсе
            feats_noisy = extract_features(pd.Series(noisy_signal), window_size).values
            X_anomaly_list.append(feats_noisy[anomaly_mask])
    
    X_normal  = np.vstack(X_normal_list)
    X_anomaly = np.vstack(X_anomaly_list) if X_anomaly_list else np.empty((0, 2))
    
    n_normal  = len(X_normal)
    n_anomaly = len(X_anomaly)
    
    # Балансировка: соотношение нормальных к аномальным = 3:1
    # Меньше чем 5:1 было раньше — делаем ещё строже, чтобы сбить агрессию модели
    target_ratio = 10
    n_normal_keep = min(n_normal, n_anomaly * target_ratio)
    
    rng = np.random.default_rng(random_state)
    idx_normal_sampled = rng.choice(n_normal, size=n_normal_keep, replace=False)
    
    X_train = np.vstack([X_normal[idx_normal_sampled], X_anomaly])
    y_train = np.concatenate([
        np.ones(n_normal_keep, dtype=int),
        np.zeros(n_anomaly,    dtype=int)
    ])
    
    # Перемешиваем
    shuffle_idx = rng.permutation(len(X_train))
    X_train, y_train = X_train[shuffle_idx], y_train[shuffle_idx]
    
    #print(f"  Обучение: {n_normal_keep} норм. + {n_anomaly} аном. "f"(соотношение {target_ratio}:1) | max_depth={max_depth}")
    
    # class_weight убираем — балансировка уже сделана руками,
    # двойной перекос делал модель слишком агрессивной
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=random_state
    )
    model.fit(X_train, y_train)
    
    return model


In [26]:
def tune_hyperparameters(df_train, df_val, labels_val, param_grid, target_metric='f1'):
    """
    Подбор гиперпараметров. 
    Модель теперь обучается на синтетических аномалиях из create_corrupted_test_bench,
    поэтому параметры percentile_diff и sensitivity удалены — они больше не нужны.
    """
    keys = param_grid.keys()
    values = (param_grid[key] for key in keys)
    combinations = [dict(zip(keys, combination)) for combination in itertools.product(*values)]
    
    print(f"Подбор гиперпараметров на группе из {len(labels_val.columns)} датчиков. "
          f"Комбинаций: {len(combinations)}\n")
    
    best_params = None
    best_score = -1.0
    best_metrics = None
    results_history = []

    sensor_cols = [c for c in df_val.columns if c != 'TimeStamp']
    
    # Считаем метрики только по зашумлённым датчикам — именно это нас интересует
    corrupted_cols = [c for c in sensor_cols if 0 in labels_val[c].values]
    print(f"Зашумлённых датчиков в val: {len(corrupted_cols)} из {len(sensor_cols)}\n")
    
    if not corrupted_cols:
        print("⚠️  Нет зашумлённых датчиков в val — метрика F1 будет бессмысленной!")
        corrupted_cols = sensor_cols  # fallback

    y_val_stacked = np.concatenate([labels_val[col].values for col in corrupted_cols])

    for i, params in enumerate(combinations):
        w_size   = params['window_size']
        m_depth  = params['max_depth']
        c_ratio  = params['corruption_ratio']
        
        # 1. Обучение на синтетических аномалиях
        model = _train_model(
            df_train_raw=df_train,
            window_size=w_size,
            percentile_diff=None,   # больше не используется внутри
            max_depth=m_depth,
            corruption_ratio=c_ratio
        )
        
        # 2. Инференс только по зашумлённым датчикам val
        X_val_list = [extract_features(
                pd.to_numeric(df_val[col], errors='coerce').fillna(0).astype(float), 
                w_size
            ).values 
            for col in corrupted_cols]
        X_val_stacked = np.vstack(X_val_list)
        
        labels_pred = model.predict(X_val_stacked).astype(int)
        
        # 3. Метрики
        metrics = compute_metrics(y_val_stacked, labels_pred)
        current_score = metrics[target_metric]
        
        results_history.append({**params, **metrics})
        
        if current_score > best_score:
            best_score = current_score
            best_params = params
            best_metrics = metrics
        
        print(f"[{i+1}/{len(combinations)}] {target_metric}: {current_score:.4f} | params: {params}")

    print("="*60)
    print(f"🏆 ЛУЧШИЕ ПАРАМЕТРЫ ПО МЕТРИКЕ '{target_metric.upper()}':")
    print(f"Параметры: {best_params}")
    print(f"Метрики:   {best_metrics}")
    print("="*60)
    
    history_df = pd.DataFrame(results_history).sort_values(by=target_metric, ascending=False)
    with open("best_model_config.json", "w", encoding='utf-8') as f:
        json.dump(best_params, f, indent=4)
        
    return best_params, history_df


In [27]:
# Задаем варианты для каждого параметра
# percentile_diff и threshold убраны: аномалии теперь генерируются реалистично,
# через ту же функцию create_corrupted_test_bench, что и val/test.
param_grid = {
    'window_size':      [10, 15, 20],
    'corruption_ratio': [0.1, 0.3, 0.5],   # Доля датчиков, которые портим при обучении
    'max_depth':        [10],
}

best_params = None

# ВРЕМЯ ПОДБОРА > 5 МИНУТ, НЕ ЗАПУСКАТЬ БЕЗ НЕОБХОДИМОСТИ

best_params, history = tune_hyperparameters(
    df_train=df_train,
    df_val=df_val,
    labels_val=labels_val,
    param_grid=param_grid,
    target_metric='f1'
)


Подбор гиперпараметров на группе из 20 датчиков. Комбинаций: 9

Зашумлённых датчиков в val: 4 из 20

[1/9] f1: 0.4734 | params: {'window_size': 10, 'corruption_ratio': 0.1, 'max_depth': 10}
[2/9] f1: 0.4804 | params: {'window_size': 10, 'corruption_ratio': 0.3, 'max_depth': 10}
[3/9] f1: 0.4778 | params: {'window_size': 10, 'corruption_ratio': 0.5, 'max_depth': 10}
[4/9] f1: 0.4720 | params: {'window_size': 15, 'corruption_ratio': 0.1, 'max_depth': 10}
[5/9] f1: 0.4818 | params: {'window_size': 15, 'corruption_ratio': 0.3, 'max_depth': 10}
[6/9] f1: 0.4863 | params: {'window_size': 15, 'corruption_ratio': 0.5, 'max_depth': 10}
[7/9] f1: 0.4072 | params: {'window_size': 20, 'corruption_ratio': 0.1, 'max_depth': 10}
[8/9] f1: 0.4136 | params: {'window_size': 20, 'corruption_ratio': 0.3, 'max_depth': 10}
[9/9] f1: 0.4362 | params: {'window_size': 20, 'corruption_ratio': 0.5, 'max_depth': 10}
🏆 ЛУЧШИЕ ПАРАМЕТРЫ ПО МЕТРИКЕ 'F1':
Параметры: {'window_size': 15, 'corruption_ratio': 0.5, 'max_d

In [28]:
def train_universal_model(df, time_column='TimeStamp', window_size=15,
                          models_dir="saved_models", corruption_ratio=0.3, max_depth=7):
    
    os.makedirs(models_dir, exist_ok=True)
    
    model = _train_model(
        df_train_raw=df,
        window_size=window_size,
        percentile_diff=None,
        max_depth=max_depth,
        corruption_ratio=corruption_ratio,
        time_column=time_column
    )
    
    joblib.dump(model, os.path.join(models_dir, "model.pkl"))
    
    c_code = m2c.export_to_c(model)
    c_code_with_info = (
        "// ML MODEL FOR ALL SENSORS\n"
        "// OUTPUT: 1.0 = Good (Normal), 0.0 = Bad (Noise/Anomaly)\n\n"
    ) + c_code
    
    with open(os.path.join(ROOT_DIR, "model.c"), "w") as f:
        f.write(c_code_with_info)
    
    print("✅ Модель сохранена.")

if best_params is None:
    with open("best_model_config.json") as file:
        best_params = json.load(file)
        print(best_params)
train_universal_model(df_train, **best_params)


✅ Модель сохранена.


In [29]:
def run_final_validation(df_test, df_labels, model_path="saved_models/model.pkl", window_size=15):
    """
    Проводит полный инференс по всем датчикам и сравнивает с истинными метками.
    Добавлена строгая проверка типов данных.
    """
    if not os.path.exists(model_path):
        print(f"❌ Ошибка: Модель {model_path} не найдена!")
        return None
    
    model = joblib.load(model_path)
    # Берем только те колонки, которые есть и в тесте, и в метках, и это не TimeStamp
    sensor_columns = [col for col in df_test.columns if col in df_labels.columns and col != 'ТР']
    
    results = []

    print(f"🚀 Начинаем финальную валидацию по {len(sensor_columns)} каналам...")

    for col in sensor_columns:
        # --- ГЛАВНЫЙ ФИКС: Принудительное преобразование в числа ---
        # errors='coerce' превратит строки в NaN, которые мы заполним нулями
        signal = pd.to_numeric(df_test[col], errors='coerce').fillna(0).astype(float)
        
        # 2. Извлечение признаков
        features = extract_features(signal, window_size) 
        X = features.values
        
        # 3. Предсказание
        y_pred = model.predict(X).astype(int)
        
        # Преобразуем и метки тоже, на всякий случай
        y_true = pd.to_numeric(df_labels[col], errors='coerce').fillna(1).astype(int).values
        
        # 4. Расчет метрик (pos_label=0 для поиска аномалий)
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, pos_label=0, zero_division=1)
        rec = recall_score(y_true, y_pred, pos_label=0, zero_division=1)
        f1 = f1_score(y_true, y_pred, pos_label=0, zero_division=1)
        
        is_corrupted = 1 if 0 in y_true else 0
        
        results.append({
            'Sensor': col,
            'Is_Corrupted': is_corrupted,
            'Accuracy': acc,
            'Precision (Anomaly)': prec,
            'Recall (Anomaly)': rec,
            'F1-Score': f1
        })

    summary_df = pd.DataFrame(results)
    corrupted_only = summary_df[summary_df['Is_Corrupted'] == 1]
    
    print("\n" + "="*60)
    print("ИТОГИ ТЕСТИРОВАНИЯ УНИВЕРСАЛЬНОЙ МОДЕЛИ")
    print("-"*60)
    print(f"Всего датчиков проверено:    {len(summary_df)}")
    print(f"Из них было зашумлено:       {len(corrupted_only)}")
    print("-"*60)
    if len(corrupted_only) > 0:
        print(f"Средний F1-Score (по шуму):  {corrupted_only['F1-Score'].mean():.4f}")
        print(f"Средний Precision (по шуму): {corrupted_only['Precision (Anomaly)'].mean():.4f}")
        print(f"Средний Recall (по шуму):    {corrupted_only['Recall (Anomaly)'].mean():.4f}")
    else:
        print("Внимание: Зашумленные датчики не найдены в выборке!")
    print(f"Средний Accuracy (общая):    {summary_df['Accuracy'].mean():.4f}")
    print("="*60)
    
    return summary_df

val_results = run_final_validation(df_test, labels_test)
if val_results is not None:
    print("\nДатчики с худшим результатом (проверьте их):")
    print(val_results.sort_values(by='F1-Score'))

🚀 Начинаем финальную валидацию по 20 каналам...

ИТОГИ ТЕСТИРОВАНИЯ УНИВЕРСАЛЬНОЙ МОДЕЛИ
------------------------------------------------------------
Всего датчиков проверено:    20
Из них было зашумлено:       4
------------------------------------------------------------
Средний F1-Score (по шуму):  0.4543
Средний Precision (по шуму): 0.3023
Средний Recall (по шуму):    0.9715
Средний Accuracy (общая):    0.9965

Датчики с худшим результатом (проверьте их):
                        Sensor  Is_Corrupted  Accuracy  Precision (Anomaly)  \
0                Т за СТ (т.6)             0  0.997785             0.000000   
7                Обороты КВД 2             0  0.999731             0.000000   
6                Т за СТ (т.7)             0  0.996711             0.000000   
11               Т за СТ (т.1)             0  0.997113             0.000000   
12               Т за СТ (т.2)             0  0.998523             0.000000   
16               Р м. на входе             0  0.999463        